# otf2viz quickstart

Reading an OTF2 (Score-P) trace and looking at it, in a handful of cells.

This notebook needs the `otf2` Python bindings for the first section only —
they ship with Score-P, not with pip. If you do not have them, skip to
**Without an OTF2 file** at the bottom; everything after that runs anywhere.

In [ ]:
import matplotlib.pyplot as plt

import otf2viz

otf2viz.__version__

## 1. Read a trace

`read_trace` takes the `traces.otf2` anchor file or any directory containing
one — a Score-P experiment directory works directly. Point `TRACE` at yours.

In [ ]:
TRACE = "../../parmix/openmp/02_pi/score-p"

trace = otf2viz.read_trace(TRACE)
trace

The `Trace` renders as a summary because it is the last expression in the
cell. For the whole measurement in one table:

In [ ]:
otf2viz.summary(trace)

## 2. The overview

One figure, four panels, one shared palette and x axis: what each thread did,
how many were busy at once, which region cost the most, and whether the load
was balanced.

In [ ]:
otf2viz.overview(trace);

## 3. The timeline on its own

`nesting="overlay"` (the default) paints nested regions over their parent.
`"lanes"` gives each call depth its own sub-lane, turning every thread's row
into a flame graph — useful when you care about the call structure.

In [ ]:
otf2viz.timeline(trace, nesting="lanes", markers=True);

## 4. Filtering is how you make a trace readable

`filter()` returns a new trace; the original is untouched. Regions that end up
unused are pruned from the legend, but locations are kept — a thread that goes
idle under a filter stays as an empty row, which is usually the point.

In [ ]:
import re

# Only the OpenMP regions, without the implicit barriers, in the first 10 ms.
work = (
    trace
    .filter(paradigms="OPENMP")
    .filter(exclude_roles=["IMPLICIT_BARRIER"])
    .select_time(0.0, 0.010)
)
work

In [ ]:
otf2viz.timeline(work);

## 5. Keeping colours stable across plots

Build one `ColorMap` from the full trace and pass it to every plot. Colour then
follows the region, not its rank, so a filter never repaints the series that
survive. Compare the two rows below — same colours, different windows.

In [ ]:
colors = otf2viz.ColorMap.from_trace(trace)

fig, (top, bottom) = plt.subplots(2, 1, figsize=(12, 5))
otf2viz.timeline(trace, ax=top, colors=colors, legend=False, title="whole run")
otf2viz.timeline(
    trace.select_time(0.005, 0.012), ax=bottom, colors=colors, title="zoomed"
)
fig.tight_layout();

Grouping by paradigm or role is a good first look at a busy trace: it has few
enough categories to fit the palette without folding anything into `other`.

In [ ]:
otf2viz.timeline(trace, color_by="role");

## 6. The numbers behind the plots

Every chart has a table. These are the exact values, and they render as HTML in
Jupyter.

In [ ]:
otf2viz.region_table(trace, top=8)

In [ ]:
otf2viz.location_table(trace)

In [ ]:
# ...or hand it to pandas, if you have it installed.
# trace.to_dataframe().head()

## 7. Interactive exploration

With `ipywidgets` installed (`pip install 'otf2viz[jupyter]'`), this gives you a
time-window slider, a call-depth cap and a region picker above the timeline.

In [ ]:
# otf2viz.interactive_timeline(trace)

## 8. Dark theme

The dark theme is a selected palette — the same hues re-stepped for a dark
surface — not an inverted light one.

In [ ]:
otf2viz.overview(trace, theme="dark");

## Without an OTF2 file

The plots work on any start/stop data. `Trace.from_events` takes
`(location, region, start, stop)` tuples and recovers the call nesting itself,
so you can sketch a schedule by hand or feed in timings from another tool.

In [ ]:
from otf2viz import Trace

sketch = Trace.from_events(
    [
        ((0, 0), "main", 0.0, 1.0),
        ((0, 0), "compute", 0.05, 0.55),
        ((0, 0), "MPI_Allreduce", 0.55, 0.95),
        ((0, 1), "compute", 0.10, 0.60),
        ((0, 1), "MPI_Allreduce", 0.60, 0.95),
        ((1, 0), "main", 0.0, 1.0),
        ((1, 0), "compute", 0.05, 0.80),
        ((1, 0), "MPI_Allreduce", 0.80, 0.95),
        ((1, 1), "compute", 0.10, 0.35),
        ((1, 1), "MPI_Allreduce", 0.35, 0.95),
    ],
    name="two ranks, two threads",
)
otf2viz.timeline(sketch);

Rank 1's first thread computes for 0.75 s while everyone else waits in the
allreduce — the load-balance view says the same thing in one glance.

In [ ]:
otf2viz.location_breakdown(sketch);